# Lab 5 Local Notebook - Task 3
Windows local version for NYCU DLP Lab5 Task 3.

這份 notebook 不使用 Colab / Kaggle / Lightning AI 路徑；預設在本機 repo 目錄執行。


## 1. Set Working Directory


In [ ]:
from pathlib import Path
import os
import sys

DEFAULT_PROJECT_ROOT = Path(r"C:\Users\innis\.vscode\vscode_nycu_deeplearning\lab5")
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "dqn.py").exists() and DEFAULT_PROJECT_ROOT.exists():
    PROJECT_ROOT = DEFAULT_PROJECT_ROOT

if not (PROJECT_ROOT / "dqn.py").exists():
    raise FileNotFoundError(f"Cannot find dqn.py from {PROJECT_ROOT}. Open this notebook from the lab5 folder or update DEFAULT_PROJECT_ROOT.")

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())
print("Python:", sys.executable)

## 2. Check Local Runtime


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")

## 3. Optional Package Check
如果缺 Atari ROM 或套件，先在 terminal 安裝；不要在已經穩定的環境裡反覆 pip install。


In [ ]:
try:
    import gymnasium as gym
    import ale_py
    import cv2
    import imageio
    print("Gymnasium / ALE / OpenCV / imageio ready")
except Exception as e:
    print("Package check failed:", repr(e))
    print("Install example:")
    print('python -m pip install gymnasium==1.1.1 ale-py opencv-python-headless imageio imageio-ffmpeg autorom "gymnasium[atari]"')
    print("Then run: AutoROM --accept-license")

## 4. Local Output Directories


In [ ]:
from pathlib import Path

LOCAL_LAB5 = PROJECT_ROOT / "local_outputs"
LOCAL_CKPT = LOCAL_LAB5 / "checkpoints"
LOCAL_LAB5.mkdir(parents=True, exist_ok=True)
LOCAL_CKPT.mkdir(parents=True, exist_ok=True)

# Keep DRIVE_* names so the shared dqn.py backup logic can reuse the same environment variable.
DRIVE_LAB5 = LOCAL_LAB5
DRIVE_CKPT = LOCAL_CKPT

print("Local output dir:", LOCAL_LAB5)
print("Local checkpoint dir:", LOCAL_CKPT)

## 5. W&B Login
本機若已經 `wandb login` 過，這格會直接使用既有 credentials。


In [ ]:
import os
import wandb

api_key = os.environ.get("WANDB_API_KEY")
if api_key:
    wandb.login(key=api_key)
else:
    wandb.login()
print("W&B ready")

---
## 6. Evaluate One Local Checkpoint
用於直接評估單一 `.pt` 檔，例如你本機根目錄裡的 `LAB5_B11107027_task3_600000.pt`。


In [ ]:
from pathlib import Path

STUDENT_ID = "B11107027"
EVAL_MODEL_PATH = PROJECT_ROOT / "LAB5_B11107027_task3_600000.pt"
EVAL_OUTPUT_DIR = PROJECT_ROOT / "eval_task3_600000_local"
EVAL_EPISODES = 20
EVAL_SEED = 0

print("Model exists:", EVAL_MODEL_PATH.exists(), EVAL_MODEL_PATH)
print("Output dir:", EVAL_OUTPUT_DIR)

In [ ]:
# Single-checkpoint evaluation: live output
import os
import shlex
import subprocess
import sys

if not EVAL_MODEL_PATH.exists():
    raise FileNotFoundError(EVAL_MODEL_PATH)

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable,
    "-u",
    "test_model.py",
    "--model-path", str(EVAL_MODEL_PATH),
    "--output-dir", str(EVAL_OUTPUT_DIR),
    "--episodes", str(EVAL_EPISODES),
    "--seed", str(EVAL_SEED),
]

print("Command:", " ".join(shlex.quote(x) for x in cmd), flush=True)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"Evaluation failed with return code {returncode}")

---
## 7. Task 3 Local Training Config
本機訓練通常比雲端慢；這格主要用來小規模測試或 resume/debug。


In [ ]:
STUDENT_ID               = "B11107027"
T3_EPISODES              = 2500

T3_BATCH_SIZE            = 32
T3_LR                    = 0.0002
T3_ADAM_EPS              = 0.00015

T3_MEMORY_SIZE           = 200000
T3_REPLAY_START          = 20000
T3_TRAIN_PER_STEP        = 1
T3_MAX_ENV_STEPS         = 1000000

T3_EPSILON_DECAY_TYPE    = "exp"
T3_EPSILON_DECAY         = 0.99998
T3_EPSILON_DECAY_STEPS   = 260000
T3_EPSILON_MIN           = 0.01

T3_TARGET_UPDATE         = 1000
T3_SOFT_TARGET_TAU       = 0.0
T3_NOOP_MAX              = 0
T3_CHECKPOINT_FREQ       = 500000
T3_RESUME                = False

T3_SAVE_DIR              = str(LOCAL_LAB5 / "results_task3_local_dueling_ref_adameps_nstep3_slowexp99998_target1000_lr2e4_1000k")
T3_WANDB_NAME            = "task3-local-dueling-ref-adameps-nstep3-slowexp99998-target1000-lr2e4-1000k"

T3_USE_PER               = True
T3_USE_DOUBLE            = True
T3_USE_DUELING           = True
T3_N_STEP                = 3

T3_PER_ALPHA             = 0.6
T3_PER_BETA              = 0.4
T3_PER_BETA_ANNEAL_STEPS = 800000

In [ ]:
# Task 3 local training: live output
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

T3_ADAM_EPS = globals().get("T3_ADAM_EPS", 1e-8)
T3_SOFT_TARGET_TAU = globals().get("T3_SOFT_TARGET_TAU", 0.0)
T3_EPSILON_DECAY = globals().get("T3_EPSILON_DECAY", 0.999999)

os.makedirs(T3_SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)

assert "local" in T3_WANDB_NAME, f"T3_WANDB_NAME must include local: {T3_WANDB_NAME}"
assert "local" in T3_SAVE_DIR, f"T3_SAVE_DIR must include local: {T3_SAVE_DIR}"

resume_flag = ""
if T3_RESUME:
    ckpt_src = Path(DRIVE_CKPT) / "task3_checkpoint.pt"
    ckpt_dst = Path(T3_SAVE_DIR) / "checkpoint.pt"
    if ckpt_src.exists():
        shutil.copy(ckpt_src, ckpt_dst)
        resume_flag = f"--resume {shlex.quote(str(ckpt_dst))}"
        print("Loaded local checkpoint")
    else:
        print("No checkpoint found, starting fresh")

flags = []
if T3_USE_PER:
    flags.append("--use-per")
if T3_USE_DOUBLE:
    flags.append("--use-double")
if T3_USE_DUELING:
    flags.append("--use-dueling")
if resume_flag:
    flags.append(resume_flag)

cmd_parts = [
    f"DRIVE_CKPT_DIR={shlex.quote(str(DRIVE_CKPT))}",
    "PYTHONUNBUFFERED=1",
    shlex.quote(sys.executable), "-u", "dqn.py",
    "--task", "3",
    "--student-id", shlex.quote(STUDENT_ID),
    "--env-name", "ALE/Pong-v5",
    "--episodes", str(T3_EPISODES),
    "--save-dir", shlex.quote(T3_SAVE_DIR),
    "--wandb-run-name", shlex.quote(T3_WANDB_NAME),
    "--batch-size", str(T3_BATCH_SIZE),
    "--memory-size", str(T3_MEMORY_SIZE),
    "--lr", str(T3_LR),
    "--adam-eps", str(T3_ADAM_EPS),
    "--epsilon-decay-type", T3_EPSILON_DECAY_TYPE,
    "--epsilon-decay", str(T3_EPSILON_DECAY),
    "--epsilon-decay-steps", str(T3_EPSILON_DECAY_STEPS),
    "--epsilon-min", str(T3_EPSILON_MIN),
    "--replay-start-size", str(T3_REPLAY_START),
    "--target-update-frequency", str(T3_TARGET_UPDATE),
    "--soft-target-tau", str(T3_SOFT_TARGET_TAU),
    "--train-per-step", str(T3_TRAIN_PER_STEP),
    "--checkpoint-freq", str(T3_CHECKPOINT_FREQ),
    "--noop-max", str(T3_NOOP_MAX),
    "--max-env-steps", str(T3_MAX_ENV_STEPS),
    "--n-step", str(T3_N_STEP),
    "--per-alpha", str(T3_PER_ALPHA),
    "--per-beta", str(T3_PER_BETA),
    "--per-beta-anneal-steps", str(T3_PER_BETA_ANNEAL_STEPS),
] + flags

cmd = " ".join(cmd_parts)
print("Command:")
print(cmd)

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1", "DRIVE_CKPT_DIR": str(DRIVE_CKPT)},
)

for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"Training failed with return code {returncode}")

---
## 8. Evaluate Milestones In T3_SAVE_DIR
只會正式評估 `T3_SAVE_DIR` 裡面的 checkpoint；找不到時會列出附近候選檔案方便排查。


In [ ]:
# Task 3 local milestone evaluation: live output
import os
import re
import shlex
import subprocess
import sys
from pathlib import Path

EVAL_EPISODES = 20
EVAL_SEED = 0
CANDIDATE_ROOTS = [PROJECT_ROOT, LOCAL_LAB5]

run_dir = Path(T3_SAVE_DIR)
run_name = run_dir.name
pattern = re.compile(rf"LAB5_{STUDENT_ID}_task3_(\d+)\.pt$")

print("Run dir:", run_dir, flush=True)
print("Run dir exists:", run_dir.exists(), flush=True)

milestones = []
for p in run_dir.glob(f"LAB5_{STUDENT_ID}_task3_*.pt"):
    m = pattern.match(p.name)
    if m:
        milestones.append(int(m.group(1)))

milestones = sorted(set(milestones))

if not milestones:
    print("\nNo milestone checkpoints found in T3_SAVE_DIR.", flush=True)
    print("Files currently in run_dir:", flush=True)
    if run_dir.exists():
        for p in sorted(run_dir.glob("*"))[:50]:
            print(" -", p.name, flush=True)
    else:
        print(" - run_dir does not exist", flush=True)

    print("\nNearby checkpoint candidates:", flush=True)
    seen = set()
    candidates = []
    for root in CANDIDATE_ROOTS:
        if root.exists():
            for p in root.glob(f"**/LAB5_{STUDENT_ID}_task3_*.pt"):
                if p not in seen:
                    seen.add(p)
                    candidates.append(p)
    candidates = sorted(candidates)
    if candidates:
        for p in candidates[:50]:
            print(" -", p, flush=True)
    else:
        print(" - none found", flush=True)

    raise FileNotFoundError(f"No milestone checkpoints found in {run_dir}")

print("Milestones found:", milestones, flush=True)

cmd_template = [
    sys.executable,
    "-u",
    "test_model.py",
    "--model-path", "<checkpoint.pt>",
    "--output-dir", "<eval_dir>",
    "--episodes", str(EVAL_EPISODES),
    "--seed", str(EVAL_SEED),
]
print("Eval command template:", " ".join(shlex.quote(x) for x in cmd_template), flush=True)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

for milestone in milestones:
    model_path = run_dir / f"LAB5_{STUDENT_ID}_task3_{milestone}.pt"
    eval_dir = run_dir / f"eval_task3_{milestone}"

    print(f"\n=== Evaluating {run_name} @ {milestone} steps ===", flush=True)
    print(f"Using model: {model_path}", flush=True)
    print(f"Output dir: {eval_dir}", flush=True)

    cmd = [
        sys.executable,
        "-u",
        "test_model.py",
        "--model-path", str(model_path),
        "--output-dir", str(eval_dir),
        "--episodes", str(EVAL_EPISODES),
        "--seed", str(EVAL_SEED),
    ]

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"Evaluation failed at {milestone} with return code {returncode}")